# BIRD-SQL Evaluation — EX & VES cho các model Ollama

Chạy `evaluation.py` (Execution Accuracy) và `evaluation_ves.py` (Valid Efficiency Score)
cho 3 model đã có kết quả: `deepseek-coder-v2`, `gemma3`, `qwen2.5-coder`.

In [6]:
import os

BIRD_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'dev.json' in files and os.path.basename(root) == 'data':
        BIRD_ROOT = os.path.dirname(root)  # thư mục cha của "data/"
        break

assert BIRD_ROOT is not None, "Không tìm thấy bird/data/dev.json trong /kaggle/input — kiểm tra lại dataset đã attach chưa"
print('BIRD_ROOT =', BIRD_ROOT)
print('dev_databases exists:', os.path.exists(f'{BIRD_ROOT}/data/dev_databases'))

import shutil
LOCAL_DB_ROOT = '/kaggle/working/dev_databases'
if not os.path.exists(LOCAL_DB_ROOT):
    shutil.copytree(f'{BIRD_ROOT}/data/dev_databases', LOCAL_DB_ROOT)
print('DBs copied:', len(os.listdir(LOCAL_DB_ROOT)))



BIRD_ROOT = /kaggle/input/datasets/thanhtrungtran/bird-dataset-result-1/bird
dev_databases exists: True
DBs copied: 11


In [7]:
import shutil, os

LOCAL_SRC = '/kaggle/working/src'
os.makedirs(LOCAL_SRC, exist_ok=True)
shutil.copy(f'{BIRD_ROOT}/llm/src/evaluation.py', LOCAL_SRC)
shutil.copy(f'{BIRD_ROOT}/llm/src/evaluation_ves.py', LOCAL_SRC)

# Kiểm tra xác nhận file đã có --iterate_num sẵn (không cần patch nữa)
has_fix = 'iterate_num' in open(f'{LOCAL_SRC}/evaluation_ves.py').read()
print('evaluation_ves.py đã có --iterate_num:', has_fix)


evaluation_ves.py đã có --iterate_num: True


In [ ]:
import subprocess, sys, re
import pandas as pd

# model hiển thị -> (thư mục w/ knowledge, thư mục w/o knowledge)
MODEL_FOLDERS = {
    "DeepSeek-Coder-V2": ("ollama_deepseekcoder_output", "ollama_deepseekcoder_0_output"),
    "Gemma3":            ("ollama_gemma3_output",        "ollama_gemma3_0_output"),
    "Qwen2.5-Coder":     ("ollama_gwen25_output",         "ollama_gwen25_0_output"),
}

GT_DIR = f'{BIRD_ROOT}/llm/data'
DIFF_JSON = f'{BIRD_ROOT}/data/dev.json'

# Kiểm tra đủ 6 folder trước khi chạy
for name, (kg_dir, nokg_dir) in MODEL_FOLDERS.items():
    for label, d in [("w/ knowledge", kg_dir), ("w/o knowledge", nokg_dir)]:
        p = f'{BIRD_ROOT}/llm/exp_result/{d}/predict_dev.json'
        print(f"{name:20} {label:15} -> {'OK' if os.path.exists(p) else 'MISSING'}  {p}")

def run_eval_script(script_name, predicted_dir_name, extra_args=None):
    cmd = [
        sys.executable, f'{LOCAL_SRC}/{script_name}',
        '--db_root_path', LOCAL_DB_ROOT + '/',
        '--predicted_sql_path', f'{BIRD_ROOT}/llm/exp_result/{predicted_dir_name}/',
        '--ground_truth_path', GT_DIR + '/',
        '--data_mode', 'dev',
        '--num_cpus', '4',
        '--meta_time_out', '30.0',
        '--mode_gt', 'gt',
        '--mode_predict', 'gpt',
        '--diff_json_path', DIFF_JSON,
    ]
    if extra_args:
        cmd += extra_args
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'[{predicted_dir_name}/{script_name}] STDERR:', result.stderr[-2000:])
    return result.stdout

SCORE_RE = re.compile(r"(accuracy|ves)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)")

def parse_score(stdout):
    m = SCORE_RE.search(stdout)
    if not m:
        return None
    _, s, mo, c, t = m.groups()
    return [float(s), float(mo), float(c), float(t)]

In [ ]:
# ex_scores[model_name][kg/nokg] = [simple, moderate, challenging, total]
ex_scores = {}
ves_scores = {}

for name, (kg_dir, nokg_dir) in MODEL_FOLDERS.items():
    ex_scores[name] = {}
    ves_scores[name] = {}
    for label, d in [("w_kg", kg_dir), ("nokg", nokg_dir)]:
        print(f"\n===== {name} ({label}) : EX =====")
        ex_out = run_eval_script('evaluation.py', d)
        print(ex_out[-500:])
        ex_scores[name][label] = parse_score(ex_out)

        print(f"\n===== {name} ({label}) : VES (iterate_num=5) =====")
        ves_out = run_eval_script('evaluation_ves.py', d, extra_args=['--iterate_num', '5'])
        print(ves_out[-500:])
        ves_scores[name][label] = parse_score(ves_out)

In [ ]:
# Bảng theo format giống paper: Model | EX w/o knowledge | EX w/ knowledge | VES w/o knowledge | VES w/ knowledge
# Chỉ lấy cột "total" (index 3) của mỗi kết quả [simple, moderate, challenging, total]

rows = {}
for name in MODEL_FOLDERS:
    rows[name] = {
        "EX_wo_knowledge": ex_scores[name]["nokg"][3],
        "EX_w_knowledge": ex_scores[name]["w_kg"][3],
        "VES_wo_knowledge": ves_scores[name]["nokg"][3],
        "VES_w_knowledge": ves_scores[name]["w_kg"][3],
    }

summary = pd.DataFrame(rows).T
summary.to_csv('/kaggle/working/summary_ex_ves.csv')
summary